# SLA Breach Prediction — Module 2: Feature Engineering
## AI-Driven Intelligent IT Operations Platform | Innodatatics Capstone — ISB AMPBA 2025W

This notebook is **Module 2 of 4** in the SLA Breach Prediction stream.

### What this notebook does
Applies the full feature engineering strategy across 7 feature groups.
The improved `SLA_03_Model_Improved` adds **3 additional feature groups** on top of what
this notebook produces — so SLA_02 remains the foundation, and the improved model extends it.

| Group | Built here (SLA_02) | Extended in SLA_03_Improved |
|---|---|---|
| 1 SLA targets | ✅ tier_priority_risk, ordinals, tightness | — |
| 2 Temporal | ✅ hour, DOW, after-hours, weekend | ➕ is_month_end, is_quarter_end, system_tickets_24h |
| 3 Asset health | ✅ age, warranty, criticality, telemetry | — |
| 4 Historical behaviour | ✅ .shift(1).expanding() for breach rates | — |
| 5 Engineer availability | ✅ exp_tier, spec_match, junior_on_p1 | ➕ eng_tickets_7d, eng_tickets_30d |
| 6 Interactions | ✅ max_risk_flag, asset_risk_burden | ➕ client_p1_rate, days_since_client_last_ticket |
| 7 Text complexity | — (not in SLA_02) | ➕ text_word_count, critical_keyword_count |

### Key note on variance filter behaviour
The leakage-safe historical features (`asset_hist_breach_rate`, `client_hist_breach_rate`,
`engineer_hist_breach_rate`) converge to ~0.5 on synthetic data → near-zero variance →
removed by Stage 1 filter in original SLA_03.

`SLA_03_Model_Improved` lowers the VT threshold from 0.0475 to 0.001 to preserve these.
They are correct features architecturally and will carry real signal on operational data.

### Outputs saved to Drive (`ticket_pkl_outputs/`)
| File | Used by |
|---|---|
| `sla_featured.pkl` | SLA_03_Improved, SLA_04 |
| `sla_feature_list.pkl` | SLA_03_Improved |
| `sla_train_test.pkl` | SLA_03_Improved |


In [1]:
# ============================================================
# COLAB + GOOGLE DRIVE SETUP — run this cell first every session
# ============================================================
from google.colab import drive
import os, sys

drive.mount("/content/drive", force_remount=False)

BASE = "/content/drive/MyDrive/Innodatatics_Capstone"
NLP  = f"{BASE}/NLP_Ticket_Analytics"
DATA = f"{BASE}/data"

os.chdir(NLP)
sys.path.insert(0, NLP)

from pathlib import Path

PKL_DIR  = Path(f"{BASE}/ticket_pkl_outputs");      PKL_DIR.mkdir(exist_ok=True)
PLOT_DIR = Path(f"{BASE}/sla_eda_plots");            PLOT_DIR.mkdir(exist_ok=True)
DASH_DIR = Path(f"{BASE}/ticket_dashboard_outputs"); DASH_DIR.mkdir(exist_ok=True)

FILES = {
    2023: Path(f"{DATA}/IT_Ops_Intern_Ready_2023.xlsx"),
    2024: Path(f"{DATA}/IT_Ops_Intern_Ready_2024.xlsx"),
    2025: Path(f"{DATA}/IT_Ops_Intern_Ready_2025.xlsx"),
}

print("✅  Drive mounted")
print(f"   Working dir : {os.getcwd()}")
print(f"   PKL dir     : {PKL_DIR}")
missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    print(f"\n⚠️  Missing: {missing}")
else:
    print("   All data files found ✅")


Mounted at /content/drive
✅  Drive mounted
   Working dir : /content/drive/MyDrive/Innodatatics_Capstone/NLP_Ticket_Analytics
   PKL dir     : /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs
   All data files found ✅


# 1. Imports & Load

In [2]:
# =========================
# 1A. Install (uncomment on first run if packages are not already included)
# =========================
# %pip install pandas numpy scikit-learn openpyxl pathlib
# %pip install lightgbm catboost xgboost shap optuna imbalanced-learn category_encoders
# Note: The additional packages are needed by SLA_03_Model_Improved — install once for the full stream.

import warnings; warnings.filterwarnings("ignore")
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

RANDOM_STATE = 42

def load_pkl(name):
    path = PKL_DIR / name
    if not path.exists():
        print(f"  ⚠️  {name} not found"); return None
    with open(path, "rb") as f: return pickle.load(f)

def save_pkl(obj, name):
    with open(PKL_DIR / name, "wb") as f: pickle.dump(obj, f)
    print(f"  ✅  Saved → {PKL_DIR / name}")

base           = load_pkl("sla_base.pkl")
asset_stats    = load_pkl("sla_asset_stats.pkl")
engineer_stats = load_pkl("sla_engineer_perf.pkl")
client_stats   = load_pkl("sla_client_stats.pkl")

if base is None:
    raise RuntimeError("sla_base.pkl not found — run SLA_01_EDA.ipynb first")

print(f"sla_base loaded: {base.shape}")
df = base.copy()
df = df.sort_values("ticket_created_timestamp").reset_index(drop=True)
print(f"Sorted by ticket_created_timestamp ✅")


sla_base loaded: (22476, 69)
Sorted by ticket_created_timestamp ✅


# 2. Feature Engineering

In [3]:
# =========================
# 2A. Leakage guard — remove post-resolution columns before any feature work
# =========================
# This list is enforced as an exclusion throughout the entire pipeline.
# Any feature derived from these columns is also excluded.
LEAKAGE_COLS = [
    "resolution_time_minutes",
    "escalation_flag",
    "ticket_reopen_flag",
    "resolution_notes",
    "first_response_time_minutes",
    "invalid_ts",                    # derived from resolution timestamps
]
print("Leakage columns excluded from all features:")
for c in LEAKAGE_COLS:
    present = c in df.columns
    print(f"  {'  present' if present else 'not found'} : {c}")


Leakage columns excluded from all features:
    present : resolution_time_minutes
    present : escalation_flag
    present : ticket_reopen_flag
    present : resolution_notes
    present : first_response_time_minutes
  not found : invalid_ts


In [4]:
# =========================
# 2B. Group 1: SLA target features
# =========================
# These encode how demanding the SLA contract is for this specific ticket.
# They are derived from SLA_RULES joined at the base level — available at creation.
# The tier × priority combination is the single most predictive structural feature
# because it determines the resolution time window against which breach is measured.

if "resolution_time_target_minutes" in df.columns:
    # Tightness: inverse of the resolution window — higher = tighter deadline
    df["sla_resolution_tightness"] = 1.0 / df["resolution_time_target_minutes"].replace(0, np.nan)

    # Response tightness
    df["sla_response_tightness"]   = 1.0 / df["response_time_target_minutes"].replace(0, np.nan)

    # Log-transformed version — reduces the extreme range across tiers/priorities
    df["log_resolution_target"]    = np.log1p(df["resolution_time_target_minutes"])

# Priority as ordinal (respects natural order P1 > P2 > P3 > P4)
priority_map = {"P1": 4, "P2": 3, "P3": 2, "P4": 1}
df["priority_ordinal"] = df["ticket_priority"].map(priority_map).fillna(2)

# Tier as ordinal
tier_map = {"Gold": 3, "Silver": 2, "Bronze": 1}
df["tier_ordinal"] = df["service_tier"].map(tier_map).fillna(1) if "service_tier" in df.columns else 1

# Critical interaction: tier × priority — captures SLA pressure in one feature
# Gold-P1 = 12, Bronze-P4 = 1
df["tier_priority_risk"] = df["tier_ordinal"] * df["priority_ordinal"]

print("Group 1 — SLA target features:")
print(df[["priority_ordinal","tier_ordinal","tier_priority_risk",
          "sla_resolution_tightness","log_resolution_target"]].describe().round(4).to_string())


Group 1 — SLA target features:
       priority_ordinal  tier_ordinal  tier_priority_risk  sla_resolution_tightness  log_resolution_target
count        22476.0000    22476.0000          22476.0000                22463.0000             22463.0000
mean             2.5028        2.0020              5.0121                    0.0012                 7.1032
std              1.1188        0.8825              3.2896                    0.0011                 0.8818
min              1.0000        1.0000              1.0000                    0.0002                 5.4848
25%              1.0000        1.0000              3.0000                    0.0003                 6.1759
50%              3.0000        2.0000              4.0000                    0.0007                 7.2731
75%              4.0000        3.0000              8.0000                    0.0021                 7.9659
max              4.0000        3.0000             12.0000                    0.0042                 8.3712


In [5]:
# =========================
# 2C. Group 2: Temporal features
# =========================
# When a ticket is created affects how quickly the operations team can respond.
# After-hours and weekend tickets have fewer engineers available.

# Ensure timestamp features are present (may have been added in EDA)
if "ticket_created_timestamp" in df.columns:
    df["created_hour"]      = df["ticket_created_timestamp"].dt.hour
    df["created_dayofweek"] = df["ticket_created_timestamp"].dt.dayofweek
    df["created_month"]     = df["ticket_created_timestamp"].dt.month
    df["created_day"]       = df["ticket_created_timestamp"].dt.day

# After-hours: before 8am or after 6pm local
df["is_after_hours"] = (
    (df["created_hour"] < 8) | (df["created_hour"] >= 18)
).astype(int)

# Weekend
df["is_weekend"] = (df["created_dayofweek"] >= 5).astype(int)

# Business hours: 9am–5pm weekdays
df["is_business_hours"] = (
    df["created_hour"].between(9, 17) & (df["created_dayofweek"] < 5)
).astype(int)

# Peak hours: 9am–12pm weekdays — highest ticket volume, potential queue buildup
df["is_peak_hours"] = (
    df["created_hour"].between(9, 12) & (df["created_dayofweek"] < 5)
).astype(int)

# High-risk timing: after-hours + high priority = maximum response pressure
df["high_risk_timing"] = (
    (df["is_after_hours"] == 1) & (df["priority_ordinal"] >= 3)
).astype(int)

print("Group 2 — Temporal features added ✅")
print(f"  After-hours tickets : {df['is_after_hours'].sum():,} ({df['is_after_hours'].mean()*100:.1f}%)")
print(f"  Weekend tickets     : {df['is_weekend'].sum():,} ({df['is_weekend'].mean()*100:.1f}%)")
print(f"  High-risk timing    : {df['high_risk_timing'].sum():,} ({df['high_risk_timing'].mean()*100:.1f}%)")


Group 2 — Temporal features added ✅
  After-hours tickets : 13,092 (58.2%)
  Weekend tickets     : 6,328 (28.2%)
  High-risk timing    : 6,543 (29.1%)


In [6]:
# =========================
# 2D. Group 3: Asset health features
# =========================
# The state of the asset at the time of ticket creation predicts how hard
# the ticket will be to resolve quickly. A degraded asset may need hardware
# replacement rather than a quick software fix.

# Asset age (in years — normalized)
if "installation_date" in df.columns and "ticket_created_timestamp" in df.columns:
    df["asset_age_days"] = (
        df["ticket_created_timestamp"] - df["installation_date"]
    ).dt.days.fillna(0).clip(lower=0)
    df["asset_age_years"] = df["asset_age_days"] / 365.25

# Warranty status at ticket creation time
if "warranty_expiry_date" in df.columns:
    df["warranty_active"] = (
        df["warranty_expiry_date"].notna() &
        df["ticket_created_timestamp"].notna() &
        (df["warranty_expiry_date"] >= df["ticket_created_timestamp"])
    ).astype(int)
    # Days until warranty expires (negative = already expired)
    df["warranty_days_remaining"] = (
        (df["warranty_expiry_date"] - df["ticket_created_timestamp"]).dt.days
    ).fillna(-999)

# Criticality as ordinal
crit_map = {"Critical": 4, "High": 3, "Medium": 2, "Low": 1}
df["criticality_ordinal"] = df["criticality_level"].map(crit_map).fillna(2) if "criticality_level" in df.columns else 2

# Telemetry stress score (pre-ticket aggregates from EDA merge)
# cpu_mean, memory_mean, disk_mean, temperature_mean are already in base
# Composite stress indicator: weighted combination of resource utilisation
tel_cols_present = [c for c in ["cpu_mean","memory_mean","disk_mean","temperature_mean"] if c in df.columns]
if tel_cols_present:
    weights = {"cpu_mean": 0.40, "memory_mean": 0.30, "disk_mean": 0.20, "temperature_mean": 0.10}
    df["telemetry_stress_score"] = sum(
        df[c].fillna(df[c].median()) * weights.get(c, 0.25)
        for c in tel_cols_present
    )
    # High error count is a strong direct failure signal
    if "error_count_sum" in df.columns:
        df["error_count_log"] = np.log1p(df["error_count_sum"].fillna(0))

# Criticality × tier interaction: critical asset for a Gold client = very high risk
df["criticality_tier_risk"] = df["criticality_ordinal"] * df["tier_ordinal"]

print("Group 3 — Asset health features added ✅")
if tel_cols_present:
    print(f"  Telemetry stress score: mean={df['telemetry_stress_score'].mean():.1f}, std={df['telemetry_stress_score'].std():.1f}")


Group 3 — Asset health features added ✅
  Telemetry stress score: mean=61.0, std=0.5


In [7]:
# =========================
# 2E. Group 4: Historical behavioural features (leakage-safe)
# =========================
# CRITICAL: All historical features use .shift(1).expanding().mean()
# This means: for row N, the value uses only rows 0..N-1.
# The current ticket's outcome is NEVER included in its own historical feature.
# This is the correct way to create behavioural features without leakage.

df = df.sort_values("ticket_created_timestamp").reset_index(drop=True)

# --- Asset-level history ---
df["asset_hist_breach_rate"] = (
    df.groupby("asset_id")["sla_num"]
    .transform(lambda s: s.shift(1).expanding().mean())
).fillna(0.5)   # default 0.5 for first ticket on any asset

df["asset_ticket_count_cumulative"] = (
    df.groupby("asset_id").cumcount()
)

# --- Client-level history ---
df["client_hist_breach_rate"] = (
    df.groupby("client_id")["sla_num"]
    .transform(lambda s: s.shift(1).expanding().mean())
).fillna(0.5)

# --- Engineer-level history ---
df["engineer_hist_breach_rate"] = (
    df.groupby("engineer_id")["sla_num"]
    .transform(lambda s: s.shift(1).expanding().mean())
).fillna(0.5)

df["engineer_hist_sla_success"] = 1 - df["engineer_hist_breach_rate"]

# Engineer's past average resolution time (expanding mean, shifted)
df["engineer_hist_avg_resolution"] = (
    df.groupby("engineer_id")["resolution_time_minutes"]
    .transform(lambda s: s.shift(1).expanding().mean())
).fillna(df["resolution_time_minutes"].median())

# Engineer's ticket count to date (experience proxy from ticket history)
df["engineer_ticket_count_cumulative"] = df.groupby("engineer_id").cumcount()

print("Group 4 — Historical behavioural features added ✅")
print(f"  asset_hist_breach_rate    : mean={df['asset_hist_breach_rate'].mean():.3f}")
print(f"  client_hist_breach_rate   : mean={df['client_hist_breach_rate'].mean():.3f}")
print(f"  engineer_hist_sla_success : mean={df['engineer_hist_sla_success'].mean():.3f}")

# =========================
# IMPORTANT NOTE FOR SLA_03_Model_Improved
# =========================
# On synthetic data with ~40% uniform breach rate, .shift(1).expanding().mean()
# converges to ~0.5 for every engineer, asset, and client.
# This means variance ≈ 0 → original SLA_03 Stage 1 filter (threshold=0.0475) removes them.
# SLA_03_Model_Improved lowers the threshold to 0.001 to preserve these features.
# They are architecturally correct — real operational data will produce genuine variance here.
print("\n⚠️  Historical breach rate features note:")
print(f"   asset_hist_breach_rate variance   : {df['asset_hist_breach_rate'].var():.6f}")
print(f"   client_hist_breach_rate variance  : {df['client_hist_breach_rate'].var():.6f}")
print(f"   engineer_hist_breach_rate variance: {df['engineer_hist_breach_rate'].var():.6f}")
print("   Near-zero variance on synthetic data — preserved by SLA_03_Improved VT threshold=0.001")


Group 4 — Historical behavioural features added ✅
  asset_hist_breach_rate    : mean=0.398
  client_hist_breach_rate   : mean=0.400
  engineer_hist_sla_success : mean=0.602

⚠️  Historical breach rate features note:
   asset_hist_breach_rate variance   : 0.004739
   client_hist_breach_rate variance  : 0.001808
   engineer_hist_breach_rate variance: 0.001739
   Near-zero variance on synthetic data — preserved by SLA_03_Improved VT threshold=0.001


In [8]:
# =========================
# 2F. Group 5: Engineer availability features
# =========================
# An engineer's experience level and specialisation match to the ticket category
# affect how quickly they can resolve it — and therefore breach risk.

# Experience tier (ordinal)
exp_bins   = [0, 2, 5, 10, 100]
exp_labels = [1, 2, 3, 4]   # junior=1, mid=2, senior=3, expert=4
df["engineer_exp_tier"] = pd.cut(
    df["experience_years"].fillna(3),
    bins=exp_bins, labels=exp_labels, right=True
).astype(float).fillna(2)

# Specialisation match: does engineer spec align with ticket category?
# Network engineer on Network ticket = match = better outcome expected
df["spec_match"] = (
    ((df["specialization"] == "Server")  & (df["ticket_category"] == "Hardware")) |
    ((df["specialization"] == "Network") & (df["ticket_category"] == "Network"))
).astype(int)

# Junior engineer on P1 ticket = high-risk mismatch
df["junior_on_p1"] = (
    (df["engineer_exp_tier"] <= 1) & (df["priority_ordinal"] == 4)
).astype(int)

# Senior engineer on simple ticket — may be fine but worth capturing
df["senior_on_p4"] = (
    (df["engineer_exp_tier"] >= 3) & (df["priority_ordinal"] == 1)
).astype(int)

print("Group 5 — Engineer availability features added ✅")
print(f"  Specialisation match: {df['spec_match'].mean()*100:.1f}% of tickets")
print(f"  Junior on P1        : {df['junior_on_p1'].sum():,} tickets")


Group 5 — Engineer availability features added ✅
  Specialisation match: 49.9% of tickets
  Junior on P1        : 884 tickets


In [9]:
# =========================
# 2G. Group 6: Interaction features
# =========================
# Domain-logic combinations that encode specific high-risk scenarios.
# These make explicit what the model would otherwise have to learn implicitly
# from the individual features — accelerating learning and improving interpretability.

# Maximum risk scenario: after-hours + P1 + Gold tier
df["max_risk_flag"] = (
    (df["is_after_hours"] == 1) &
    (df["priority_ordinal"] == 4) &
    (df["tier_ordinal"] == 3)
).astype(int)

# Asset age × criticality: old critical assets are harder to fix quickly
df["asset_age_criticality"] = df["asset_age_years"] * df["criticality_ordinal"] if "asset_age_years" in df.columns else 0

# Historical breach burden on asset × current SLA tightness
df["asset_risk_burden"] = (
    df["asset_hist_breach_rate"] * df.get("sla_resolution_tightness", pd.Series(0.001, index=df.index))
)

# Client historical pattern × tier: Gold client with poor breach history = very high risk
df["client_tier_risk"] = df["client_hist_breach_rate"] * df["tier_ordinal"]

# Specialisation mismatch on high priority: wrong engineer type for urgent ticket
df["spec_mismatch_high_prio"] = (
    (df["spec_match"] == 0) & (df["priority_ordinal"] >= 3)
).astype(int)

print("Group 6 — Interaction features added ✅")
print(f"  Max risk flag (after-hours P1 Gold): {df['max_risk_flag'].sum():,} tickets")
print(f"  Spec mismatch on high priority     : {df['spec_mismatch_high_prio'].sum():,} tickets")


Group 6 — Interaction features added ✅
  Max risk flag (after-hours P1 Gold): 1,240 tickets
  Spec mismatch on high priority     : 5,734 tickets


In [10]:
# =========================
# 2H. Final feature inventory
# =========================
# Collect the complete feature set, verify no leakage columns are included,
# and categorise into categorical and numeric pools.

# Leakage exclusion list — enforced here
LEAKAGE_EXCLUDE = set([
    "resolution_time_minutes", "escalation_flag", "ticket_reopen_flag",
    "resolution_notes", "first_response_time_minutes", "invalid_ts",
    "sla_breach_flag", "sla_num",              # targets
    "year_ticket_id", "ticket_id",             # IDs
    "ticket_created_timestamp", "ticket_close_timestamp",  # raw timestamps
    "contract_start_date", "contract_end_date",
    "installation_date", "warranty_expiry_date",
    "timestamp", "engineer_name", "client_name",
    "ticket_description", "combined_text",     # NLP-only fields
    "resolution_notes", "reco_text",
    "year",                                    # source year — proxy for time trends
])

# Categorical features
# engineer_id and client_id are included here for TargetEncoder in SLA_03_Improved.
# They are high-cardinality (20 and 25 unique values) — OHE gives sparse, weak columns,
# but TargetEncoder replaces them with a single numeric column = mean(breach) per category.
FEATURE_CATS = [c for c in [
    "ticket_category", "issue_type", "ticket_priority",
    "ticket_channel", "service_tier", "industry_sector", "region",
    "device_type", "manufacturer", "criticality_level",
    "specialization", "shift_type", "support_level",
    "engineer_id", "client_id",   # added for TargetEncoder in SLA_03_Improved
] if c in df.columns and c not in LEAKAGE_EXCLUDE]

# Numeric features
FEATURE_NUMS = [c for c in [
    # SLA target group
    "priority_ordinal", "tier_ordinal", "tier_priority_risk",
    "sla_resolution_tightness", "sla_response_tightness", "log_resolution_target",
    "response_time_target_minutes",
    # Temporal group
    "created_hour", "created_dayofweek", "created_month",
    "is_after_hours", "is_weekend", "is_business_hours", "is_peak_hours",
    "high_risk_timing",
    # Asset group
    "asset_age_days", "asset_age_years", "warranty_active", "warranty_days_remaining",
    "criticality_ordinal", "telemetry_stress_score", "error_count_log",
    "criticality_tier_risk",
    "cpu_mean", "memory_mean", "disk_mean", "temperature_mean", "latency_mean",
    "packet_loss_mean", "failure_flag_mean",
    # Historical group
    "asset_hist_breach_rate", "asset_ticket_count_cumulative",
    "client_hist_breach_rate",
    "engineer_hist_breach_rate", "engineer_hist_sla_success",
    "engineer_hist_avg_resolution", "engineer_ticket_count_cumulative",
    # Engineer group
    "engineer_exp_tier", "spec_match", "junior_on_p1", "senior_on_p4",
    "experience_years",
    # Interaction group
    "max_risk_flag", "asset_age_criticality", "asset_risk_burden",
    "client_tier_risk", "spec_mismatch_high_prio",
] if c in df.columns and c not in LEAKAGE_EXCLUDE]

print(f"Feature inventory:")
print(f"  Categorical features : {len(FEATURE_CATS)}")
print(f"  Numeric features     : {len(FEATURE_NUMS)}")
print(f"  Total features       : {len(FEATURE_CATS) + len(FEATURE_NUMS)}")
print(f"\nCategorical: {FEATURE_CATS}")
print(f"\nNumeric (first 20): {FEATURE_NUMS[:20]}")

# Verify no leakage
overlap = (set(FEATURE_CATS) | set(FEATURE_NUMS)) & LEAKAGE_EXCLUDE
if overlap:
    print(f"\n❌  LEAKAGE DETECTED: {overlap}")
else:
    print(f"\n✅  No leakage columns in feature set")


Feature inventory:
  Categorical features : 15
  Numeric features     : 47
  Total features       : 62

Categorical: ['ticket_category', 'issue_type', 'ticket_priority', 'ticket_channel', 'service_tier', 'industry_sector', 'region', 'device_type', 'manufacturer', 'criticality_level', 'specialization', 'shift_type', 'support_level', 'engineer_id', 'client_id']

Numeric (first 20): ['priority_ordinal', 'tier_ordinal', 'tier_priority_risk', 'sla_resolution_tightness', 'sla_response_tightness', 'log_resolution_target', 'response_time_target_minutes', 'created_hour', 'created_dayofweek', 'created_month', 'is_after_hours', 'is_weekend', 'is_business_hours', 'is_peak_hours', 'high_risk_timing', 'asset_age_days', 'asset_age_years', 'warranty_active', 'warranty_days_remaining', 'criticality_ordinal']

✅  No leakage columns in feature set


# 3. Train/Test Split (Time-Based)

In [11]:
# =========================
# 3A. Time-based split
# =========================
# MANDATORY: 2023-2024 train → 2025 test.
# A random split would allow future tickets into the training set,
# giving artificially inflated performance metrics.
# TimeSeriesSplit is used for cross-validation within the training set.

train_df = df[df["year"].isin([2023, 2024])].copy()
test_df  = df[df["year"] == 2025].copy()

print(f"Train set : {len(train_df):,} rows (2023–2024)")
print(f"Test set  : {len(test_df):,} rows (2025)")
print(f"\nTrain breach rate : {train_df['sla_num'].mean()*100:.1f}%")
print(f"Test breach rate  : {test_df['sla_num'].mean()*100:.1f}%")
print(f"\nFeatures: {len(FEATURE_CATS)} categorical + {len(FEATURE_NUMS)} numeric")


Train set : 15,002 rows (2023–2024)
Test set  : 7,474 rows (2025)

Train breach rate : 39.4%
Test breach rate  : 40.1%

Features: 15 categorical + 47 numeric


# 4. Save Outputs

In [12]:
# =========================
# 4A. Save all feature engineering outputs
# =========================
feature_list = {"cats": FEATURE_CATS, "nums": FEATURE_NUMS, "leakage_exclude": list(LEAKAGE_EXCLUDE)}

save_pkl(df,           "sla_featured.pkl")
save_pkl(feature_list, "sla_feature_list.pkl")
save_pkl({"train": train_df, "test": test_df}, "sla_train_test.pkl")

print("\n✅  Feature engineering complete → run SLA_03_Model.ipynb")
print(f"   Total engineered features : {len(FEATURE_CATS) + len(FEATURE_NUMS)}")
print(f"   Train rows : {len(train_df):,}  |  Test rows : {len(test_df):,}")

# Save a note of what SLA_03_Improved will add on top
additional_features_note = {
    "sla_02_groups": ["SLA_targets","Temporal","Asset_health","Historical_behaviour",
                      "Engineer_availability","Interaction_features"],
    "sla_03_improved_adds": {
        "engineer_workload": ["eng_tickets_7d","eng_tickets_30d","eng_tickets_today","eng_cumulative_tickets"],
        "client_behavioral": ["client_tickets_30d","client_p1_rate","days_since_client_last_ticket","client_cumulative_tickets"],
        "temporal_congestion": ["is_month_end","is_quarter_end","is_year_end","system_tickets_24h","is_high_volume_day"],
        "text_complexity": ["text_word_count","has_critical_keywords","critical_keyword_count","issue_combo_frequency"],
    },
    "variance_filter_fix": "SLA_03_Improved uses VT threshold=0.001 (vs 0.0475 original) to preserve historical features",
    "encoding_upgrade": "TargetEncoder for engineer_id, client_id (vs OHE only in original)",
}
save_pkl(additional_features_note, "sla_features_note.pkl")
print("\n✅  SLA_02 complete → run SLA_03_Model_Improved.ipynb (recommended)")
print("   (SLA_03_Model_Improved extends these features with 3 additional groups)")


  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/sla_featured.pkl
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/sla_feature_list.pkl
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/sla_train_test.pkl

✅  Feature engineering complete → run SLA_03_Model.ipynb
   Total engineered features : 62
   Train rows : 15,002  |  Test rows : 7,474
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/sla_features_note.pkl

✅  SLA_02 complete → run SLA_03_Model_Improved.ipynb (recommended)
   (SLA_03_Model_Improved extends these features with 3 additional groups)
